In [ ]:
import os
import torch
import nibabel as nib
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import torchio as tio
from nilearn.masking import compute_brain_mask
from transformers import Dinov2Model
from PIL import Image
from torchvision import transforms

In [ ]:
# Function to clip background slices in all three axes
def clip_background_slices(image_data, threshold=0):
    non_empty_slices_axial = [i for i in range(image_data.shape[2]) if np.max(image_data[:, :, i]) > threshold]
    clipped_image_axial = image_data[:, :, non_empty_slices_axial]
    non_empty_slices_coronal = [i for i in range(clipped_image_axial.shape[1]) if np.max(clipped_image_axial[:, i, :]) > threshold]
    clipped_image_coronal = clipped_image_axial[:, non_empty_slices_coronal, :]
    non_empty_slices_sagittal = [i for i in range(clipped_image_coronal.shape[0]) if np.max(clipped_image_coronal[i, :, :]) > threshold]
    clipped_image_sagittal = clipped_image_coronal[non_empty_slices_sagittal, :, :]
    return clipped_image_sagittal

# Function to normalize image data
def normalize_image(image_data):
    transform = tio.RescaleIntensity(out_min_max=(0, 1))
    image_data = transform(torch.tensor(image_data).unsqueeze(0)).squeeze(0).numpy()
    return image_data

# Function to visualize a few slices
def visualize_slices(image_data, title, num_slices=5):
    fig, axs = plt.subplots(1, num_slices, figsize=(15, 5))
    print(image_data.shape)
    for i in range(num_slices):
        slice_idx = image_data.shape[2] // (num_slices + 1) * (i + 1)
        axs[i].imshow(image_data[:, :, slice_idx], cmap='gray')
        axs[i].set_title(f'Slice {slice_idx}')
    plt.suptitle(title)
    plt.show()

# Function to preprocess NIfTI file and visualize each step
def preprocess_nifti_with_visualization(file_path, target_size=(224, 224)):
    try:
        # Load NIfTI image
        nifti_image = nib.load(file_path)
        image_data = nifti_image.get_fdata()
        
        # Convert sagittal view to axial view
        axial_image = np.transpose(image_data, (0, 2, 1))
        #visualize_slices(axial_image, "Axial View of Original Image")

        # Skull stripping
        brain_mask = compute_brain_mask(nifti_image, threshold=0.1).get_fdata()

        # Check if brain mask is empty
        if np.sum(brain_mask) == 0:
            print(f"Skipping {file_path} due to empty brain mask.")
            return None
        
        # Transpose brain mask to match axial image
        brain_mask = np.transpose(brain_mask, (0, 2, 1))
        stripped_image = axial_image * brain_mask
        #visualize_slices(stripped_image, "Skull Stripped Image")

        # Resample to (1, 1, 1)
        resample = tio.Resample((1, 1, 1), image_interpolation='linear')
        subject = tio.Subject(image=tio.ScalarImage(tensor=stripped_image[np.newaxis, ...]))
        resampled_subject = resample(subject)
        image_data_resampled = resampled_subject.image.data.numpy().squeeze()
        #visualize_slices(image_data_resampled, "Resampled Image")

        # Normalize image data
        normalized_image = normalize_image(image_data_resampled)
        #visualize_slices(normalized_image, "Normalized Image")

        # Clip background slices
        image_data_clipped = clip_background_slices(normalized_image)
        #visualize_slices(image_data_clipped, "Clipped Image")

        # Extract 50 slices from the middle
        middle = image_data_clipped.shape[2] // 2
        start = max(0, middle - 25)
        end = min(image_data_clipped.shape[2], middle + 25)
        extracted_slices = image_data_clipped[:, :, start:end]
        #visualize_slices(extracted_slices, "Extracted Middle Slices")

        # Resize slices to target size
        resized_slices = []
        for slice_idx in range(extracted_slices.shape[2]):
            slice_data = extracted_slices[:, :, slice_idx]
            resized_slice = F.interpolate(torch.tensor(slice_data).unsqueeze(0).unsqueeze(0).float(), size=target_size, mode='bilinear', align_corners=False).squeeze(0).squeeze(0)
            resized_slices.append(resized_slice.numpy())  
        resized_slices = np.stack(resized_slices, axis=-1)  
        #visualize_slices(resized_slices, "Resized Slices")

        return resized_slices
    
    except Exception as e:
        print(f"Error processing {file_path}: {str(e)}")
        return None

In [ ]:
class SplineLinear(nn.Linear):
    def __init__(self, in_features: int, out_features: int, init_scale: float = 0.1, **kw) -> None:
        self.init_scale = init_scale
        super().__init__(in_features, out_features, bias=False, **kw)

    def reset_parameters(self) -> None:
        nn.init.xavier_uniform_(self.weight)


class ReflectionalSwitchFunction(nn.Module):
    def __init__(
            self,
            grid_min: float = -2.,
            grid_max: float = 2.,
            num_grids: int = 8,
            exponent: int = 2,
            denominator: float = 0.33,  
    ):
        super().__init__()
        grid = torch.linspace(grid_min, grid_max, num_grids)
        self.grid = torch.nn.Parameter(grid, requires_grad=False)
        self.denominator = denominator  
        self.inv_denominator = 1 / self.denominator  

    def forward(self, x):
        diff = (x[..., None] - self.grid)
        diff_mul = diff.mul(self.inv_denominator)
        diff_tanh = torch.tanh(diff_mul)
        diff_pow = -diff_tanh.mul(diff_tanh)
        diff_pow += 1
        return diff_pow  


class FasterKANLayer(nn.Module):
    def __init__(
            self,
            input_dim: int,
            output_dim: int,
            grid_min: float = -2.,
            grid_max: float = 2.,
            num_grids: int = 8,
            exponent: int = 2,
            denominator: float = 0.33,
            use_base_update: bool = True,
            base_activation=F.silu,
            spline_weight_init_scale: float = 0.1,
    ) -> None:
        super().__init__()
        self.layernorm = nn.LayerNorm(input_dim)
        self.rbf = ReflectionalSwitchFunction(grid_min, grid_max, num_grids, exponent, denominator)
        self.spline_linear = SplineLinear(input_dim * num_grids, output_dim, spline_weight_init_scale)

    def forward(self, x, time_benchmark=False):
        if not time_benchmark:
            spline_basis = self.rbf(self.layernorm(x)).view(x.shape[0], -1)
        else:
            spline_basis = self.rbf(x).view(x.shape[0], -1)
        
        ret = self.spline_linear(spline_basis)
        return ret

class FasterKAN(nn.Module):
    def __init__(
            self,
            layers_hidden: List[int],
            grid_min: float = -2.,
            grid_max: float = 2.,
            num_grids: int = 8,
            exponent: int = 2,
            denominator: float = 0.33,
            use_base_update: bool = True,
            base_activation=F.silu,
            spline_weight_init_scale: float = 0.667,
    ) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            FasterKANLayer(
                in_dim, out_dim,
                grid_min=grid_min,
                grid_max=grid_max,
                num_grids=num_grids,
                exponent=exponent,
                denominator=denominator,
                use_base_update=use_base_update,
                base_activation=base_activation,
                spline_weight_init_scale=spline_weight_init_scale,
            ) for in_dim, out_dim in zip(layers_hidden[:-1], layers_hidden[1:])
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
class kanBlock(nn.Module):
    def __init__(self, dim, num_heads=8, hdim_kan=192, mlp_ratio=4., qkv_bias=False, qk_scale=None, drop=0., attn_drop=0.,
                 drop_path=0., act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super().__init__()
        self.norm1 = norm_layer(dim)
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, dropout=attn_drop)
        self.drop_path = nn.Identity() if drop_path <= 0. else nn.Dropout(drop_path)
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.kan = FasterKAN(
            layers_hidden=[dim, hdim_kan, dim],
            grid_min=-2.,
            grid_max=2.,
            num_grids=8,
            exponent=2,
            denominator=0.33,
            use_base_update=True,
            base_activation=act_layer(),
            spline_weight_init_scale=0.1
        )

    def forward(self, x):
        b, t, d = x.shape
        x_norm1 = self.norm1(x) 
        attn_output, _ = self.attn(x_norm1, x_norm1, x_norm1)
        x = x + self.drop_path(attn_output)
        
        x_norm2 = self.norm2(x).reshape(-1, d)
        kan_output = self.kan(x_norm2).reshape(b, t, d) 
        x = x + self.drop_path(kan_output)
        
        return x

In [ ]:
class DinoV2KAN(nn.Module):
    def __init__(self, num_classes=10):
        super(DinoV2KAN, self).__init__()
        
        # Load the pre-trained DINOv2 model
        self.dino = Dinov2Model.from_pretrained('facebook/dinov2-base')
        
        # Freeze DINOv2 layers
        for param in self.dino.parameters():
            param.requires_grad = False
        
        # Get the number of features from the DINOv2 model
        dino_feature_size = self.dino.config.hidden_size
        
        # Initialize kanBlock
        self.kan_block = kanBlock(
            dim=dino_feature_size,
            num_heads=12,
            hdim_kan=768,
            mlp_ratio=4.,
            qkv_bias=False,
            qk_scale=None,
            drop=0.1,
            attn_drop=0.1,
            drop_path=0.1,
            act_layer=nn.GELU,
            norm_layer=nn.LayerNorm
        )
        
        # Final classification layer
        self.classifier = nn.Linear(dino_feature_size, num_classes)
    
    def forward(self, x):
        # Pass the input through the pre-trained DINOv2 model
        outputs = self.dino(x)
        features = outputs.last_hidden_state.mean(dim=1)  
        
        # Pass the features through kanBlock
        x = self.kan_block(features.unsqueeze(1)).squeeze(1)
        
        # Pass through the final classifier
        x = self.classifier(x)
        
        return x

In [ ]:
import os
import torch
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
from model import DinoV2KAN  # Assume your model is saved here
from your_preprocessing_module import preprocess_nifti_with_visualization  # Your function

# --- Configuration ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint_path = "checkpoints/fold_1_best_model.pth"
num_classes = 3
image_path = "path/to/image.jpg"  # Replace dynamically via CLI or call

# --- Model Setup ---
model = DinoV2KAN(num_classes=num_classes)
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.ToTensor(),  # Converts to [C,H,W]
    transforms.Resize((224, 224)),
])

def predict_from_slices(slices, model, device):
    preds = []
    probs_list = []
    for i in range(slices.shape[2]):
        slice_img = slices[:, :, i]
        img = Image.fromarray((slice_img * 255).astype(np.uint8)).convert("RGB")
        input_tensor = transform(img).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(input_tensor)
            probs = F.softmax(output, dim=1)
            pred = torch.argmax(probs, dim=1).item()
        preds.append(pred)
        probs_list.append(probs.cpu().numpy())
    probs_array = np.mean(np.stack(probs_list), axis=0)
    final_pred = int(np.argmax(probs_array))
    return final_pred, probs_array

if image_path.endswith(('.nii', '.nii.gz')):
    slices = preprocess_nifti_with_visualization(image_path)
    if slices is None:
        print("Preprocessing failed.")
    else:
        pred_class, probs = predict_from_slices(slices, model, device)
else:
    img = Image.open(image_path)
    if img.mode != 'RGB':
        img = img.convert('L').convert('RGB')  # Ensure 3 channels
    input_tensor = transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        output = model(input_tensor)
        probs = F.softmax(output, dim=1)
        pred_class = torch.argmax(probs, dim=1).item()
        probs = probs.cpu().numpy()

# --- Output ---
print(f"Predicted class: {pred_class}")
print(f"Probabilities: {probs}")
